In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


def metricas_classificacao(y_true, y_pred, target_names):
    report = classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )
    tabela = pd.DataFrame(report).transpose()
    tabela = tabela.drop(index='accuracy', errors='ignore')
    tabela = tabela[['precision', 'recall', 'f1-score', 'support']]
    tabela['support'] = tabela['support'].astype(int)
    return tabela

# 1. Carregar os dados
df = pd.read_csv('../data/Dataset-Mental-Disorders.csv')

# 2. Limpeza essencial
X = df.drop(['Patient Number', 'Expert Diagnose'], axis=1)
y = df['Expert Diagnose']

# 3. Transformar texto em numeros
le = LabelEncoder()
for col in X.columns:
    X[col] = le.fit_transform(X[col])

y = le.fit_transform(y)
target_names = le.classes_

print('Distribuicao das classes:')
print(pd.Series(le.inverse_transform(y)).value_counts().sort_index())

# 4. Divisao treino/teste estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Modelo
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

# 6. Predicao e resultados
y_pred = modelo.predict(X_test)

print('=== RELATORIO DE CLASSIFICACAO ===')
print(metricas_classificacao(y_test, y_pred, target_names).to_string(float_format=lambda value: f'{value:.4f}'))

# 7. Visualizacao
plt.figure(figsize=(10, 7))
sns.heatmap(
    confusion_matrix(y_test, y_pred),
    annot=True,
    fmt='d',
    xticklabels=target_names,
    yticklabels=target_names,
    cmap='Blues'
)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusao - Diagnosticos Mentais')
plt.show()
